In [1]:
import pandas as pd
from transformers import AutoTokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained("nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16")

In [3]:
data = pd.read_csv("../data/solved/train-cot.csv")

In [4]:
data

,id,prompt,answer,prompt_eda,label,generated_cot,computed_answer,is_correct
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,Task type: bit_rules\n\nSelected rules:\nout[0...,10010111,True
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,Task type: bit_rules\nReason: at least one out...,NaN,False
2,0031df9c,"In Alice's Wonderland, a secret bit manipulati...",00110100,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,Task type: macro\n\nSelected rule:\nrotate rig...,00110100,True
3,004ef7c7,"In Alice's Wonderland, a secret bit manipulati...",11111111,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,Task type: bit_rules\nReason: at least one out...,NaN,False
4,00754598,"In Alice's Wonderland, a secret bit manipulati...",11101111,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,Task type: bit_rules\n\nSelected rules:\nout[0...,11101111,True
...,...,...,...,...,...,...,...,...
7940,ff2e376c,"In Alice's Wonderland, the gravitational const...",190.03,"In Alice's Wonderland, the gravitational const...",gravitational,"WARNING: This is Wonderland gravity, NOT Earth...",190.03,True
7941,ff85238e,"In Alice's Wonderland, the gravitational const...",57.61,"In Alice's Wonderland, the gravitational const...",gravitational,"WARNING: This is Wonderland gravity, NOT Earth...",57.61,True
7942,ff90228b,"In Alice's Wonderland, the gravitational const...",32.39,"In Alice's Wonderland, the gravitational const...",gravitational,"WARNING: This is Wonderland gravity, NOT Earth...",32.39,True
7943,ff9540e2,"In Alice's Wonderland, the gravitational const...",107.7,"In Alice's Wonderland, the gravitational const...",gravitational,"WARNING: This is Wonderland gravity, NOT Earth...",107.70,False


In [5]:
data["token_len"] = data.generated_cot.apply(tokenizer.encode).apply(len)
data["token_len"].describe()

count    7945.000000
mean      458.642794
std       250.895952
min        26.000000
25%       196.000000
50%       541.000000
75%       657.000000
max      1045.000000
Name: token_len, dtype: float64

In [6]:
data.groupby("label").token_len.describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
bit manipulation,1602.0,512.061174,204.239147,26.0,514.0,574.0,644.0,739.0
conversion to diff numeral system,1576.0,156.190990,25.320367,110.0,133.0,156.0,174.0,225.0
encryption,1576.0,777.158629,91.775663,531.0,714.0,776.5,840.0,1045.0
gravitational,1597.0,601.629931,47.983251,524.0,545.0,600.0,655.0,670.0
unit conversion,1594.0,245.817440,34.100198,188.0,224.0,237.0,276.0,307.0


In [7]:
data

,id,prompt,answer,prompt_eda,label,generated_cot,computed_answer,is_correct,token_len
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,Task type: bit_rules\n\nSelected rules:\nout[0...,10010111,True,586
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,Task type: bit_rules\nReason: at least one out...,NaN,False,26
2,0031df9c,"In Alice's Wonderland, a secret bit manipulati...",00110100,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,Task type: macro\n\nSelected rule:\nrotate rig...,00110100,True,173
3,004ef7c7,"In Alice's Wonderland, a secret bit manipulati...",11111111,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,Task type: bit_rules\nReason: at least one out...,NaN,False,26
4,00754598,"In Alice's Wonderland, a secret bit manipulati...",11101111,"In Alice's Wonderland, a secret bit manipulati...",bit manipulation,Task type: bit_rules\n\nSelected rules:\nout[0...,11101111,True,589
...,...,...,...,...,...,...,...,...,...
7940,ff2e376c,"In Alice's Wonderland, the gravitational const...",190.03,"In Alice's Wonderland, the gravitational const...",gravitational,"WARNING: This is Wonderland gravity, NOT Earth...",190.03,True,605
7941,ff85238e,"In Alice's Wonderland, the gravitational const...",57.61,"In Alice's Wonderland, the gravitational const...",gravitational,"WARNING: This is Wonderland gravity, NOT Earth...",57.61,True,606
7942,ff90228b,"In Alice's Wonderland, the gravitational const...",32.39,"In Alice's Wonderland, the gravitational const...",gravitational,"WARNING: This is Wonderland gravity, NOT Earth...",32.39,True,596
7943,ff9540e2,"In Alice's Wonderland, the gravitational const...",107.7,"In Alice's Wonderland, the gravitational const...",gravitational,"WARNING: This is Wonderland gravity, NOT Earth...",107.70,False,605


In [ ]:
target_labels = ['bit manipulation']


correct_col = 'is_correct' if 'is_correct' in data.columns else 'is correct'

filtered_data = data[(data['label'].isin(target_labels)) & (data[correct_col] == True)]

sampled_data = filtered_data.groupby('label').sample(n=5, random_state=24424).reset_index(drop=True)

for index, row in sampled_data.iterrows():
    print(f"=== Категория: {row['label']} | ID: {row['id']} ===")
    print("--- Промпт (начало) ---")
    print(str(row['prompt']) + "...\n")
    
    print("--- Решение ---")
    cot_val = row.get('generated_cot', row.get('generated cot', 'Отсутствует'))
    print(cot_val)
    #
    #print("\n--- Вычисленный ответ ---")
    #ans_val = row.get('computed_answer', row.get('computed answer', 'Отсутствует'))
    #print(ans_val)
#
    #print("\n--- Истинный ответ ---")
    #ans_val = row.get('answer', row.get('answer', 'Отсутствует'))
    #print(ans_val)
    #
    #print("\n" + "="*80 + "\n")

In [9]:
filtered_data.token_len.describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: token_len, dtype: float64

In [10]:
len(filtered_data[filtered_data.token_len <= 7500])#.loc[ 751].generated_cot

0

In [11]:
print(filtered_data.loc[9416].generated_cot)

KeyError: 9416

In [ ]:
print(filtered_data.loc[8488].prompt)

In [ ]:
text = """In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
[&+/` = '!
<\-<\ = \
`|-'' = -<|
/&*?\ = &/|
//-?? = -``
Now, determine the result for: &&+&`..."""